# Main Priority Model (Decision Layer)

Combines outputs from plumbing, electrical, and structural models into a final priority score.

Priority logic:
- Risk levels: Low=1, Medium=2, High=3
- Urgency = 60 - days_to_failure
- Impact weight: girls_toilet=3, classroom=2, storage=1
- Priority = sum(risk x urgency x weight), normalized to 0-100

Run all previous notebooks first.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, joblib, json, warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Load All Models

In [13]:
plumbing_model = joblib.load('../models/plumbing_model.pkl')
plumbing_features = joblib.load('../models/plumbing_features.pkl')
print(f'Plumbing model loaded ({len(plumbing_features)} features)')

electrical_model = joblib.load('../models/electrical_model.pkl')
electrical_features = joblib.load('../models/electrical_features.pkl')
print(f'Electrical model loaded ({len(electrical_features)} features)')

structural_model = joblib.load('../models/structural_model.pkl')
structural_features = joblib.load('../models/structural_features.pkl')
structural_le = joblib.load('../models/structural_label_encoder.pkl')
print(f'Structural model loaded ({len(structural_features)} features)')

df_full = pd.read_csv('../data/preprocessed/full_preprocessed.csv')
print(f'\nFull dataset: {df_full.shape}')

Plumbing model loaded (24 features)
Electrical model loaded (26 features)
Structural model loaded (25 features)

Full dataset: (50000, 48)


In [14]:
# show saved metrics
for name in ['plumbing', 'electrical', 'structural']:
    with open(f'../models/{name}_metrics.json') as f:
        m = json.load(f)
    print(f'\n{name.upper()}:')
    print(f'  Accuracy: {m.get("accuracy", "N/A")}')
    if 'f1_score' in m:
        print(f'  F1:       {m["f1_score"]}')
    if 'weighted_f1' in m:
        print(f'  F1 (wt):  {m["weighted_f1"]}')


PLUMBING:
  Accuracy: 1.0
  F1:       1.0

ELECTRICAL:
  Accuracy: 0.9997
  F1:       0.9996

STRUCTURAL:
  Accuracy: 0.9991
  F1 (wt):  0.9991


## 2. Priority Logic Functions

In [15]:
def prob_to_risk(prob):
    if prob >= 0.7: return 'High'
    if prob >= 0.4: return 'Medium'
    return 'Low'

def risk_value(risk):
    return {'Low': 1, 'Medium': 2, 'High': 3}.get(risk, 1)

def severity_to_risk(sev):
    return {'Safe': 'Low', 'Warning': 'Medium', 'Danger': 'High'}.get(sev, 'Low')

def impact_weight(category, girls_school):
    if category == 'plumbing' and girls_school:
        return 3
    if category in ['plumbing', 'electrical']:
        return 2
    return 1

def priority_score(risk_val, urgency, weight):
    raw = risk_val * max(0, urgency) * weight
    return min(100, round(raw / (3 * 60 * 3) * 100, 2))

def priority_label(score):
    if score >= 75: return 'Critical'
    if score >= 50: return 'High'
    if score >= 25: return 'Medium'
    return 'Low'

print('Priority functions defined')

Priority functions defined


## 3. predict_all() Function

In [16]:
def predict_all(input_data, verbose=True):
    """
    Runs all 3 models and computes final priority.
    
    Returns dict with:
    - risks per category
    - days to failure
    - final priority score
    - explanation
    """
    if isinstance(input_data, dict):
        input_data = pd.Series(input_data)
    
    category = input_data.get('category', 'unknown')
    girls = int(input_data.get('girls_school', 0))
    students = float(input_data.get('num_students', 0))
    dtf = float(input_data.get('days_to_failure', 60))
    
    results = {'school_id': input_data.get('school_id', 'N/A'), 'category': category}
    
    # plumbing prediction
    try:
        p_in = input_data[plumbing_features].fillna(0).values.reshape(1, -1)
        p_prob = plumbing_model.predict_proba(p_in)[0][1]
        p_risk = prob_to_risk(p_prob)
    except:
        p_prob, p_risk = 0.0, 'Low'
    
    # electrical prediction
    try:
        e_in = input_data[electrical_features].fillna(0).values.reshape(1, -1)
        e_prob = electrical_model.predict_proba(e_in)[0][1]
        e_risk = prob_to_risk(e_prob)
    except:
        e_prob, e_risk = 0.0, 'Low'
    
    # structural prediction
    try:
        s_in = input_data[structural_features].fillna(0).values.reshape(1, -1)
        s_pred = structural_model.predict(s_in)[0]
        s_label = structural_le.inverse_transform([s_pred])[0]
        s_risk = severity_to_risk(s_label)
    except:
        s_label, s_risk = 'Safe', 'Low'
    
    results['category_risks'] = {
        'plumbing': {'probability': round(float(p_prob), 4), 'risk': p_risk},
        'electrical': {'probability': round(float(e_prob), 4), 'risk': e_risk},
        'structural': {'severity': s_label, 'risk': s_risk},
    }
    
    # compute final priority
    urgency = max(0, 60 - dtf)
    
    cat_scores = []
    for cat, risk in [('plumbing', p_risk), ('electrical', e_risk), ('structural', s_risk)]:
        rv = risk_value(risk)
        wt = impact_weight(cat, girls)
        sc = priority_score(rv, urgency, wt)
        cat_scores.append({'category': cat, 'risk_value': rv, 'weight': wt, 'score': sc})
    
    total = sum(c['score'] for c in cat_scores)
    
    # weight the actual category more
    if category in ['plumbing', 'electrical', 'structural']:
        idx = {'plumbing': 0, 'electrical': 1, 'structural': 2}[category]
        final = min(100, round(cat_scores[idx]['score'] * 0.6 + total * 0.4 / 3, 2))
    else:
        final = min(100, round(total / 3 * 1.5, 2))
    
    label = priority_label(final)
    
    results['priority'] = {
        'score': final,
        'label': label,
        'days_to_failure': round(dtf, 2),
        'urgency': round(urgency, 2),
        'breakdown': cat_scores,
    }
    
    # build explanation
    parts = []
    risk_map = {'plumbing': p_risk, 'electrical': e_risk, 'structural': s_risk}
    high = [c for c, r in risk_map.items() if r == 'High']
    med = [c for c, r in risk_map.items() if r == 'Medium']
    if high: parts.append(f"HIGH risk in {', '.join(high)}")
    if med: parts.append(f"MEDIUM risk in {', '.join(med)}")
    if girls: parts.append('girls school (priority boost)')
    if urgency > 40: parts.append(f'urgent ({dtf:.0f} days to failure)')
    if students > 1000: parts.append(f'high impact ({students:.0f} students)')
    
    results['explanation'] = f"{label} priority (score: {final}/100) - {'; '.join(parts)}."
    
    if verbose:
        print(f'\nSchool {results["school_id"]} ({category})')
        print(f'  Plumbing:    {p_risk} (prob: {p_prob:.3f})')
        print(f'  Electrical:  {e_risk} (prob: {e_prob:.3f})')
        print(f'  Structural:  {s_risk} ({s_label})')
        print(f'  Priority:    {final}/100 ({label})')
        print(f'  {results["explanation"]}')
    
    return results

print('predict_all() defined')

predict_all() defined


## 4. Test Predictions

In [17]:
# test with one sample from each category
for cat in ['plumbing', 'electrical', 'structural']:
    sample = df_full[df_full['category'] == cat].iloc[0]
    predict_all(sample)
    print('---')


School 1000 (plumbing)
  Plumbing:    Low (prob: 0.008)
  Electrical:  Low (prob: 0.001)
  Structural:  Medium (Warning)
  Priority:    30.2/100 (Medium)
  Medium priority (score: 30.2/100) - MEDIUM risk in structural; girls school (priority boost); urgent (0 days to failure).
---

School 1000 (electrical)
  Plumbing:    High (prob: 0.828)
  Electrical:  High (prob: 0.935)
  Structural:  Medium (Warning)
  Priority:    65.48/100 (High)
  High priority (score: 65.48/100) - HIGH risk in plumbing, electrical; MEDIUM risk in structural; girls school (priority boost); urgent (-0 days to failure).
---

School 1000 (structural)
  Plumbing:    Low (prob: 0.096)
  Electrical:  Low (prob: 0.097)
  Structural:  Medium (Warning)
  Priority:    22.14/100 (Low)
  Low priority (score: 22.14/100) - MEDIUM risk in structural; urgent (0 days to failure).
---


## 5. Batch Predictions

In [18]:
sample_df = df_full.sample(n=500, random_state=42)

batch = []
for _, row in sample_df.iterrows():
    r = predict_all(row, verbose=False)
    batch.append({
        'school_id': r['school_id'],
        'category': r['category'],
        'plumbing_risk': r['category_risks']['plumbing']['risk'],
        'electrical_risk': r['category_risks']['electrical']['risk'],
        'structural_severity': r['category_risks']['structural']['severity'],
        'priority_score': r['priority']['score'],
        'priority_label': r['priority']['label'],
    })

results_df = pd.DataFrame(batch)
print(f'Batch done: {len(results_df)} records')
print(f'\nPriority distribution:')
print(results_df['priority_label'].value_counts())
results_df.head(10)

Batch done: 500 records

Priority distribution:
priority_label
Low         297
High         99
Medium       88
Critical     16
Name: count, dtype: int64


,school_id,category,plumbing_risk,electrical_risk,structural_severity,priority_score,priority_label
0,2347,electrical,High,High,Warning,61.13,High
1,1387,electrical,Low,Low,Warning,22.18,Low
2,1009,plumbing,Low,Low,Warning,30.32,Medium
3,1510,structural,High,High,Warning,34.79,Medium
4,2584,structural,High,High,Warning,39.17,Medium
5,2709,electrical,Low,Low,Warning,21.93,Low
6,1445,plumbing,High,High,Warning,61.25,High
7,2978,structural,Low,Low,Warning,21.82,Low
8,1170,plumbing,Low,Low,Warning,22.29,Low
9,2485,electrical,Low,Low,Warning,23.72,Low


In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

# Add actual failure flag to results
results_df['actual_failure'] = sample_df['failure_within_30_days'].values
results_df['priority_binary'] = (results_df['priority_score'] >= 50).astype(int)

print('=== MODEL ACCURACY VALIDATION ===\n')

# 1. Individual Model Accuracies (from saved metrics)
print('1. Individual Domain Model Performance:')
print('-' * 60)
for name in ['plumbing', 'electrical', 'structural']:
    try:
        with open(f'../models/{name}_metrics.json') as f:
            m = json.load(f)
        print(f'\n{name.upper()} Model:')
        print(f'  Accuracy:  {m.get("accuracy", "N/A")}')
        print(f'  Precision: {m.get("precision", "N/A")}')
        print(f'  Recall:    {m.get("recall", "N/A")}')
        print(f'  F1-Score:  {m.get("f1_score", "N/A")}')
        if 'roc_auc' in m:
            print(f'  ROC-AUC:   {m.get("roc_auc", "N/A")}')
    except Exception as e:
        print(f'  Error loading {name} metrics: {e}')

# 2. Combined Priority Model Accuracy
print('\n\n2. Combined Priority Model Accuracy:')
print('-' * 60)

accuracy = accuracy_score(results_df['actual_failure'], results_df['priority_binary'])
precision = precision_score(results_df['actual_failure'], results_df['priority_binary'], zero_division=0)
recall = recall_score(results_df['actual_failure'], results_df['priority_binary'], zero_division=0)
f1 = f1_score(results_df['actual_failure'], results_df['priority_binary'], zero_division=0)

print(f'Total Samples Tested: {len(results_df)}')
print(f'Actual Failures: {results_df["actual_failure"].sum()} ({results_df["actual_failure"].mean():.2%})')
print(f'\nAccuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')

# 3. Confusion Matrix
cm = confusion_matrix(results_df['actual_failure'], results_df['priority_binary'])
print(f'\nConfusion Matrix:')
print(f'  True Negatives:  {cm[0,0]}')
print(f'  False Positives: {cm[0,1]}')
print(f'  False Negatives: {cm[1,0]}')
print(f'  True Positives:  {cm[1,1]}')

# 4. Per-Category Accuracy
print('\n\n3. Accuracy by Infrastructure Category:')
print('-' * 60)
for cat in results_df['category'].unique():
    cat_data = results_df[results_df['category'] == cat]
    cat_acc = accuracy_score(cat_data['actual_failure'], cat_data['priority_binary'])
    cat_prec = precision_score(cat_data['actual_failure'], cat_data['priority_binary'], zero_division=0)
    cat_recall = recall_score(cat_data['actual_failure'], cat_data['priority_binary'], zero_division=0)
    print(f'\n{cat.upper()}:')
    print(f'  Samples:   {len(cat_data)}')
    print(f'  Accuracy:  {cat_acc:.4f}')
    print(f'  Precision: {cat_prec:.4f}')
    print(f'  Recall:    {cat_recall:.4f}')

# 5. Priority Label Accuracy
print('\n\n4. Priority Label to Actual Outcome Mapping:')
print('-' * 60)
for label in ['Low', 'Medium', 'High', 'Critical']:
    label_data = results_df[results_df['priority_label'] == label]
    if len(label_data) > 0:
        failure_rate = label_data['actual_failure'].mean()
        count = len(label_data)
        failures = label_data['actual_failure'].sum()
        print(f'{label:8s}: {count:3d} samples ({failures:2d} failures) - {failure_rate:.2%} actual failure rate')


=== MODEL ACCURACY VALIDATION ===

1. Individual Domain Model Performance:
------------------------------------------------------------

PLUMBING Model:
  Accuracy:  1.0
  Precision: 1.0
  Recall:    1.0
  F1-Score:  1.0
  ROC-AUC:   1.0

ELECTRICAL Model:
  Accuracy:  0.9997
  Precision: 0.9992
  Recall:    1.0
  F1-Score:  0.9996
  ROC-AUC:   1.0

STRUCTURAL Model:
  Accuracy:  0.9991
  Precision: N/A
  Recall:    N/A
  F1-Score:  N/A


2. Combined Priority Model Accuracy:
------------------------------------------------------------
Total Samples Tested: 500
Actual Failures: 169 (33.80%)

Accuracy:  0.8920 (89.20%)
Precision: 1.0000
Recall:    0.6805
F1-Score:  0.8099

Confusion Matrix:
  True Negatives:  331
  False Positives: 0
  False Negatives: 54
  True Positives:  115


3. Accuracy by Infrastructure Category:
------------------------------------------------------------

ELECTRICAL:
  Samples:   156
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000

PLUMBING:
  Samples

## 6. Accuracy Validation & Model Performance

In [20]:
## 7. Performance Visualization

## 8. Example Input/Output JSON

In [21]:
# pick a high-priority girls school example
sample = df_full[(df_full['failure_within_30_days']==1) & (df_full['girls_school']==1)].iloc[0]
result = predict_all(sample, verbose=False)

print('Example Input (key fields):')
inp = {
    'school_id': int(sample.get('school_id', 0)),
    'category': str(sample.get('category', '')),
    'girls_school': int(sample.get('girls_school', 0)),
    'num_students': int(sample.get('num_students', 0)),
    'building_age': float(sample.get('building_age', 0)),
    'condition_score': float(sample.get('condition_score', 0)),
    'water_leak': int(sample.get('water_leak', 0)),
    'wiring_exposed': int(sample.get('wiring_exposed', 0)),
    'crack_width_mm': float(sample.get('crack_width_mm', 0)),
    'days_to_failure': float(sample.get('days_to_failure', 0)),
}
print(json.dumps(inp, indent=2))

print('\nExample Output:')
print(json.dumps(result, indent=2, default=str))

Example Input (key fields):
{
  "school_id": 1000,
  "category": "electrical",
  "girls_school": 1,
  "num_students": 1,
  "building_age": 1.5330491779976636,
  "condition_score": 0.0066446916648983,
  "water_leak": 0,
  "wiring_exposed": 0,
  "crack_width_mm": 1.1070225691325115,
  "days_to_failure": -0.3317649465635063
}

Example Output:
{
  "school_id": "1000",
  "category": "electrical",
  "category_risks": {
    "plumbing": {
      "probability": 0.8281,
      "risk": "High"
    },
    "electrical": {
      "probability": 0.9349,
      "risk": "High"
    },
    "structural": {
      "severity": "Warning",
      "risk": "Medium"
    }
  },
  "priority": {
    "score": 65.48,
    "label": "High",
    "days_to_failure": -0.33,
    "urgency": 60.33,
    "breakdown": [
      {
        "category": "plumbing",
        "risk_value": 3,
        "weight": 3,
        "score": 100
      },
      {
        "category": "electrical",
        "risk_value": 3,
        "weight": 2,
        "score":

## 9. API-Ready Function

In [22]:
def predict_api(input_json):
    """API wrapper. Pass a dict, get back structured response."""
    try:
        result = predict_all(pd.Series(input_json), verbose=False)
        return {
            'success': True,
            'data': {
                'school_id': result['school_id'],
                'risks': {
                    'plumbing': result['category_risks']['plumbing']['risk'],
                    'electrical': result['category_risks']['electrical']['risk'],
                    'structural': result['category_risks']['structural']['severity'],
                },
                'priority': result['priority'],
                'explanation': result['explanation'],
            }
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

api_result = predict_api(sample.to_dict())
print('API response:')
print(json.dumps(api_result, indent=2, default=str))

API response:
{
  "success": true,
  "data": {
    "school_id": 1000,
    "risks": {
      "plumbing": "High",
      "electrical": "High",
      "structural": "Warning"
    },
    "priority": {
      "score": 65.48,
      "label": "High",
      "days_to_failure": -0.33,
      "urgency": 60.33,
      "breakdown": [
        {
          "category": "plumbing",
          "risk_value": 3,
          "weight": 3,
          "score": 100
        },
        {
          "category": "electrical",
          "risk_value": 3,
          "weight": 2,
          "score": 67.04
        },
        {
          "category": "structural",
          "risk_value": 2,
          "weight": 1,
          "score": 22.35
        }
      ]
    },
    "explanation": "High priority (score: 65.48/100) - HIGH risk in plumbing, electrical; MEDIUM risk in structural; girls school (priority boost); urgent (-0 days to failure)."
  }
}


## 10. Save Pipeline Config

In [23]:
config = {
    'pipeline': 'InfraRakshak Predictive Maintenance',
    'version': '1.0.0',
    'models': {
        'plumbing': {'path': 'models/plumbing_model.pkl', 'type': 'RandomForest', 'target': 'failure_within_30_days'},
        'electrical': {'path': 'models/electrical_model.pkl', 'type': 'RandomForest', 'target': 'failure_within_30_days'},
        'structural': {'path': 'models/structural_model.pkl', 'type': 'RandomForest', 'target': 'structural_severity'},
    },
    'priority_logic': {
        'risk_weights': {'Low': 1, 'Medium': 2, 'High': 3},
        'impact_weights': {'girls_toilet': 3, 'classroom': 2, 'storage': 1},
        'urgency_formula': '60 - days_to_failure',
        'thresholds': {'Critical': 75, 'High': 50, 'Medium': 25, 'Low': 0},
    }
}

with open('../models/pipeline_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Pipeline config saved')
print('\nAll files:')
for f in os.listdir('../models'):
    print(f'  {f}')

Pipeline config saved

All files:
  plumbing_metrics.json
  plumbing_features.pkl
  scale_columns.pkl
  electrical_model.pkl
  pipeline_config.json
  structural_features.pkl
  structural_metrics.json
  standard_scaler.pkl
  plumbing_baseline_lr.pkl
  structural_label_encoder.pkl
  electrical_features.pkl
  structural_model.pkl
  electrical_metrics.json
  plumbing_model.pkl
